# Customer Churn Analysis — IBM Telco Dataset

**Business question:** which customers leave, why do they leave, and what should a retention team do about it?

**Dataset:** the IBM Telco Customer Churn dataset (public, on Kaggle). It was created by IBM for educational use and may not reflect any real telecom company's customer base. This is a portfolio project on that public sample dataset — the findings describe this dataset, not a real company.

**Tools:** Python (pandas, matplotlib). The same key questions are answered in SQL in `../sql/churn_segments.sql`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load the data

One row per customer. Download the CSV from Kaggle first (see `../data/DATA_SOURCE.md`).

In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)

(7043, 21)


## 2. Fix TotalCharges before anything else

TotalCharges is stored as text, not as a number. 11 rows have a blank value — and all 11 are zero-tenure, brand-new customers who have not been billed yet. So the fix is: convert the column to numeric, and treat blanks as 0. If you skip this, the column stays as text and any sums or averages on it silently break.

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Rows with blank TotalCharges:', int(df['TotalCharges'].isna().sum()))
print('Tenure of those rows:', df.loc[df['TotalCharges'].isna(), 'tenure'].unique())
df['TotalCharges'] = df['TotalCharges'].fillna(0)

Rows with blank TotalCharges: 11
Tenure of those rows: [0]


## 3. How many customers churn?

1,869 of 7,043 customers churned — 26.54%.

In [4]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).mul(100).round(2))

Yes    1869
No     5174
Name: Churn, dtype: int64
Yes    26.54
No     73.46
Name: Churn, dtype: float64


## 4. Churn by contract — the biggest gap in the data

Month-to-month customers churn at 42.7%, one-year at 11.3%, two-year at 2.8%. A month-to-month customer is about 15x more likely to churn than a two-year customer. There are 3,875 month-to-month customers, and 1,655 of them churned.

In [5]:
contract_rate = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
print(contract_rate.round(1))
print('Month-to-month customers:', int((df['Contract'] == 'Month-to-month').sum()))
print('Month-to-month churned:', int(((df['Contract'] == 'Month-to-month') & (df['Churn'] == 'Yes')).sum()))

Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
Name: Churn, dtype: float64
Month-to-month customers: 3875
Month-to-month churned: 1655


## 5. Churn by internet service

Fiber optic customers churn at 41.9%, DSL at 19.0%. Same product category, very different outcome — that points to a service or support problem with fiber, not a pricing problem.

In [6]:
internet_rate = df.groupby('InternetService')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
print(internet_rate.loc[['Fiber optic', 'DSL']].round(1))

InternetService
Fiber optic    41.9
DSL            19.0
Name: Churn, dtype: float64


## 6. Churn by payment method

Electronic check is the worst way to pay, from a retention view: 45.3% churn. Automatic bank transfer and credit card are around 15–17%. Customers who pay automatically stay.

In [7]:
payment_rate = df.groupby('PaymentMethod')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
print(payment_rate.loc[['Electronic check']].round(1))

PaymentMethod
Electronic check    45.3
Name: Churn, dtype: float64


## 7. Churn by tenure — the first year is where customers are lost

47.4% of customers churn in their first 12 months, and about 52% in the first 6 months. After 4 years, churn drops to about 8–9.5%. Median tenure is 10 months for customers who churned vs 38 months for customers who stayed.

In [8]:
first_year = df[df['tenure'] <= 12]
print('Churn rate, first 12 months:', round((first_year['Churn'] == 'Yes').mean() * 100, 1))
first_6 = df[df['tenure'] <= 6]
print('Churn rate, first 6 months:', round((first_6['Churn'] == 'Yes').mean() * 100, 1))
print(df.groupby('Churn')['tenure'].median())

Churn rate, first 12 months: 47.4
Churn rate, first 6 months: 52.0
Churn
No     38.0
Yes    10.0
Name: tenure, dtype: float64


## 8. The highest-risk segment

Put the three risk factors together. Month-to-month customers in their first 12 months churn at 51.4%. And among month-to-month customers, those without TechSupport churn at 50.4% vs 30.7% for those with TechSupport — so support really does keep people.

In [9]:
risk = df[(df['Contract'] == 'Month-to-month') & (df['tenure'] <= 12)]
print('Month-to-month, first 12 months:', round((risk['Churn'] == 'Yes').mean() * 100, 1))
mm = df[df['Contract'] == 'Month-to-month']
print(mm.groupby('TechSupport')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100).loc[['No', 'Yes']].round(1))

Month-to-month, first 12 months: 51.4
TechSupport
No     50.4
Yes    30.7
Name: Churn, dtype: float64


## 9. Seniors and monthly charges

Senior citizens churn at 41.7% vs 23.6% for other customers. Churned customers also paid more: $74.44 a month on average vs $61.27 for customers who stayed.

In [10]:
print(df.groupby('SeniorCitizen')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100).round(1))
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))

SeniorCitizen
0    23.6
1    41.7
Name: Churn, dtype: float64
Churn
No     61.27
Yes    74.44
Name: MonthlyCharges, dtype: float64


## 10. Charts

In [11]:
contract_rate.plot(kind='bar')
plt.ylabel('Churn rate (%)')
plt.title('Churn rate by contract type')
plt.tight_layout()
plt.show()

In [12]:
internet_rate.loc[['Fiber optic', 'DSL']].plot(kind='bar')
plt.ylabel('Churn rate (%)')
plt.title('Churn rate by internet service')
plt.tight_layout()
plt.show()

## 11. What the retention team should do

1. **Nudge month-to-month customers onto one-year or two-year contracts.** 42.7% churn vs 2.8% is the biggest gap in the data — even a small shift here saves the most customers.
2. **Protect the first year.** With 47.4% churning in the first 12 months and about 52% in the first 6, onboarding check-ins in months 1–6 matter most.
3. **Look at fiber optic service quality.** Fiber customers churn at 41.9% vs 19.0% for DSL. That is a service or support problem to investigate, not a pricing problem.
4. **Move electronic-check payers to autopay, and offer TechSupport to month-to-month customers.** Electronic check churns at 45.3% vs about 15–17% for automatic payments, and month-to-month customers without TechSupport churn at 50.4% vs 30.7% with it.

## 12. Limitations

- This is a sample dataset made for learning. The patterns describe this dataset; they are associations, not proof of cause. For example, fiber customers may churn more because of the service, or because of who buys fiber — this data cannot tell those apart.
- There is no complaint, outage, or customer-service data here, so the "why" behind fiber churn and early churn is a question for follow-up.
- Revenue at risk: $139,131 in monthly recurring revenue from churned customers, per a published analysis of this dataset.
- What I would do next: join in complaint and service-call data if it existed, and check whether the contract effect still holds after accounting for tenure.